# Streaming Triggers and Output Modes: Append, Complete, and Update

**Module**: 09_streaming  
**Notebook**: 03_triggers_and_output_modes.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Deep dive into Triggers (Default, ProcessingTime, Once, AvailableNow) and Output Modes (Append, Complete, Update).
- **Why does this topic exist?**: Matching the correct output mode to your query type is mandatory; invalid combinations fail at query planning.
- **What problem does it solve?**: Attempting to run Complete mode on a non-aggregated stream or Append mode on global aggregations.
- **Where is it used in real-world Data Engineering?**: Production scheduling of hourly micro-batches using Trigger.AvailableNow.

---

## 2. Output Mode Matrix
- **Append**: Emits only newly inserted rows. (Required for file sinks; disallowed for global non-watermarked aggregations).
- **Complete**: Re-emits the entire updated result table. (Required for global aggregations; unsupported on file sinks).
- **Update**: Emits only the rows updated since the previous trigger.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Triggers_Modes_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
rate_stream = spark.readStream.format("rate").option("rowsPerSecond", 10).load()
print("Rate stream initialized.")

---

## Experiment: Trigger.AvailableNow Execution

- **1. What are we testing?**: Execute a stream using Trigger.AvailableNow() and observe that it shuts down automatically.
- **2. What do we expect?**: Processes all available rows and terminates cleanly.

In [ ]:
q = (
    rate_stream
    .writeStream
    .format("memory")
    .queryName("avail_now_sink")
    .trigger(availableNow=True)
    .start()
)
q.awaitTermination()
print("Query terminated automatically after Trigger.AvailableNow!")
spark.sql("SELECT count(*) as total_rows FROM avail_now_sink").show()

**3. What happened?**: Stream ran to completion and shut down automatically.  
**4. Why did it happen?**: Trigger.AvailableNow processes pending micro-batches and terminates.  
**5. Practical Engineering Lesson**: Use Trigger.AvailableNow for cost-effective incremental batch ETL scheduled via Airflow.

---

## Practical Exercises

### Exercise 1 (Beginner): Fixed Interval Trigger
**Problem Statement**: Configure trigger with processingTime='5 seconds'.  
**Input / Context**: trigger

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Complete Mode Aggregation
**Problem Statement**: Run a streaming group count in Complete mode to memory sink.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Trigger.Once vs AvailableNow
**Problem Statement**: Explain why AvailableNow replaced Once.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Fixed Interval Trigger):
```python
print('trigger(processingTime="5 seconds")')
```

#### Solution 2 (Complete Mode Aggregation):
```python
print('rate_stream.groupBy().count().writeStream.outputMode("complete")')
```

#### Solution 3 (Trigger.Once vs AvailableNow):
```python
print('Trigger.Once processed only 1 micro-batch; AvailableNow processes multiple batches until caught up.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Modes**: Append (new rows), Complete (all rows), Update (modified rows).
- **AvailableNow**: Cost-effective serverless batch processing.

### Essential Technical Terms
- `Trigger.AvailableNow`: Process all pending data and stop.
- `OutputMode`: Defines what is emitted to the sink.

### Common Engineering Mistakes
- **Complete mode on File Sink**: File sink does not support rewriting full tables; requires Append mode.

### Interview Questions on This Topic
- **Q: When should you use Trigger.AvailableNow()?**
  - *A*: When you want incremental streaming semantics but run as an hourly or daily scheduled batch to save compute costs.

### What to Study Next
Open 04_watermarks_and_late_data.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')